In [ ]:
#@title 1. Setup - run once { display-mode: "form" }
#@markdown Tick to keep downloads in your Google Drive (asks you to sign in).
#@markdown Unticked, they go to `/content/downloads`, wiped when the runtime ends.
use_drive = False  #@param {type:"boolean"}
drive_folder = "yt-downloads"  #@param {type:"string"}
#@markdown Leave blank to keep the saved key (one is generated the first time).
api_key = ""  #@param {type:"string"}

import os, subprocess

REPO, DIR = "https://github.com/freelancermeer/ytmeer.git", "/content/ytmeer"

!apt-get -qq install -y ffmpeg aria2 > /dev/null 2>&1
!pip -q install "yt-dlp>=2026.7.4" "curl_cffi>=0.10,<0.16" "gradio>=6,<7"

if os.path.isdir(f"{DIR}/.git"):
    subprocess.run(["git", "-C", DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO, DIR], check=True)

if use_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTDIR = f"/content/drive/MyDrive/{drive_folder.strip().strip('/') or 'yt-downloads'}"
else:
    OUTDIR = "/content/downloads"
OUTDIR = os.path.abspath(OUTDIR)
os.makedirs(OUTDIR, exist_ok=True)

# Saved beside the code, not in the environment, so cell 2 - and any server it
# starts - still has them after a runtime restart.
with open(f"{DIR}/.api_outdir", "w") as f:
    f.write(OUTDIR)
if api_key.strip():
    # Pending until cell 2 next starts a server, so a server still downloading
    # stays reachable with the key it was started with.
    fd = os.open(f"{DIR}/.api_key_new", os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    with os.fdopen(fd, "w") as f:
        f.write(api_key.strip() + "\n")
print("Setup done. Downloads go to", OUTDIR)

In [ ]:
#@title 2. Start the API { display-mode: "form" }
#@markdown Tick only to replace a server that is still downloading - its jobs are stopped first.
force_restart = False  #@param {type:"boolean"}

import hashlib, json, os, signal, socket, subprocess, sys, time, urllib.error, urllib.request

DIR = "/content/ytmeer"


def _read(name):
    try:
        with open(f"{DIR}/{name}") as f:
            return f.read().strip()
    except OSError:
        return ""


def _get(port, path):
    """The JSON the API on this port answers, or None."""
    req = urllib.request.Request(f"http://127.0.0.1:{port}/api/{path}",
                                 headers={"X-API-Key": _read(".api_key")})
    try:
        with urllib.request.urlopen(req, timeout=3) as r:
            return json.load(r)
    except Exception:
        return None


FOLDER = _read(".api_outdir")
if not FOLDER:
    raise RuntimeError("Run cell 1 first - it chooses the download folder.")
PORT = int(_read(".api_port") or 8000)
code = hashlib.sha1()
for name in ("api.py", "downloader.py"):
    with open(f"{DIR}/{name}", "rb") as f:
        code.update(f.read())
code = code.hexdigest()[:12]

new_key = _read(".api_key_new")
if new_key and new_key == _read(".api_key"):
    os.remove(f"{DIR}/.api_key_new")
    new_key = ""


def _running(what):
    return subprocess.run(["pgrep", "-f", f"{DIR}/{what}[.]py"], capture_output=True, text=True).stdout.split()


ping, health = _get(PORT, "ping"), _get(PORT, "health")
# A server that is alive but does not answer counts as busy, not as gone.
busy = ping.get("busy") if ping else bool(_running("api"))
current = (bool(health) and health["code"] == code and health["default_folder"] == FOLDER
           and not new_key)

if not current and busy and not force_restart:
    # Never throw away a download in progress because cell 1 changed something.
    print(f"NOTE: the API on port {PORT} is still downloading, so it was left as it is;\n"
          "      cell 1's changes apply once it is replaced. Re-run this cell when its\n"
          "      jobs are done, or tick force_restart.")
    if not health:
        raise RuntimeError("...and it cannot be reached. Tick force_restart to replace it.")
elif not current:
    # Stop the server first - it stops its own jobs as it shuts down - then any
    # downloader still going, left by a server that died without doing so. Each
    # downloader leads its own process group, so that reaches its yt-dlp too.
    subprocess.run(["pkill", "-f", f"{DIR}/api[.]py"])
    for _ in range(120):
        if not _running("api"):
            break
        time.sleep(0.25)
    for pid in _running("downloader"):
        try:
            os.killpg(int(pid), signal.SIGINT)
        except OSError:
            pass
    for _ in range(120):
        if not _running("downloader"):
            break
        time.sleep(0.25)
    if new_key:
        os.replace(f"{DIR}/.api_key_new", f"{DIR}/.api_key")
    with socket.socket() as s:            # the port may belong to something else
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)   # as uvicorn binds
        try:
            s.bind(("127.0.0.1", PORT))
        except OSError:
            s.bind(("127.0.0.1", 0))
            PORT = s.getsockname()[1]
    with open(f"{DIR}/.api_port", "w") as f:
        f.write(str(PORT))
    env = {k: v for k, v in os.environ.items() if not k.startswith("YTDL_")}
    with open(f"{DIR}/api.log", "a") as log:
        subprocess.Popen([sys.executable, f"{DIR}/api.py", "--port", str(PORT), "--outdir", FOLDER,
                          "--share"],
                         stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    for _ in range(240):                  # the public link takes a few seconds to open
        if _get(PORT, "health"):
            break
        time.sleep(0.5)
    else:
        with open(f"{DIR}/api.log") as log:
            print(log.read()[-3000:])
        raise RuntimeError("The API did not start - its log is above.")

API_KEY = _read(".api_key")
API_URL = f"http://127.0.0.1:{PORT}/api"
HEADERS = {"X-API-Key": API_KEY}
info = _get(PORT, "health") or {}
PUBLIC_URL = f"{info['public_url']}/api" if info.get("public_url") else None
if PUBLIC_URL:
    print(f"  public  {PUBLIC_URL}")
    print(f"  docs    {PUBLIC_URL}/docs      (Authorize, then paste the key)")
else:
    print(f"  public  none - Gradio could not open a share link; see {DIR}/api.log")
print(f"  local   {API_URL}      (for cells in this notebook)")
print(f"  key     {API_KEY}      header: X-API-Key")
print(f"  folder  {info.get('default_folder', FOLDER)}")
print('''
Use the public URL from anywhere, with the key in an X-API-Key header. In a cell
below, API_URL, PUBLIC_URL, API_KEY and HEADERS are already set. For example:

    import requests, time

    def call(method, path, **kw):
        r = requests.request(method, API_URL + path, headers=HEADERS, timeout=60, **kw)
        if not r.ok:
            raise RuntimeError(f"{r.status_code}: {r.text[:300]}")
        return r.json()

    job = call("POST", "/jobs", json={"links": ["https://www.youtube.com/@SomeChannel"],
                                      "views": 1000, "limit": 3})
    since = 0
    while True:
        page = call("GET", f"/jobs/{job['job_id']}/videos", params={"since": since})
        for v in page["videos"]:
            if v["status"] in ("ok", "skipped"):
                print(v["title"], v["files"]["video"], v["files"]["transcript"])
        since = page["next"]
        if page["done"]:
            break
        time.sleep(5)
    job = call("GET", f"/jobs/{job['job_id']}")
    print(job["state"], job["error"], job["failures"])
''')